# 0. Load python's packages
We need:
1. Pandas (https://pandas.pydata.org/docs/user_guide/index.html)
2. Numpy (https://numpy.org/doc/stable/user/index.html)
3. Scipy.optimize (https://docs.scipy.org/doc/scipy/tutorial/optimize.html)
4. Matplotlib.pyplot (https://matplotlib.org/stable/users/index.html)

In [ ]:
import pandas as pd
import numpy as np
import scipy.optimize as spo
import matplotlib.pyplot as plt

# 1. Data & files management - Pandas
1. Familiarise with the data-file: you have the format and the contents of the file containing the data you want to read
2. Read files and store them as Pandas' DataFrame
3. Check your DataFrame was created correctly
4. Read output files generated by other software: MAESTRO

### 1.2 (& 1.3) Read files and store them as Pandas' DataFrame

In [ ]:
# Open a .csv file - no comments, no headers
filename = "data/data0.csv"
d0 = pd.read_csv(filename, sep=',')
print(d0)

In [ ]:
# Assign a key to each column to control how to access them
d0 = pd.read_csv(filename, sep=',', names=['a','b','c'])
print(d0)
print("*"*25)
print(d0['a'])

In [ ]:
# If headers already in the correct format
filename = "data/data0bis.csv"
d0 = pd.read_csv(filename, sep=',')
print(d0)

In [ ]:
# Open a .csv file - with comments
filename = "data/data1.csv"
d1 = pd.read_csv(filename, sep=',', comment='#', names=['e','pd','pd_std'])
print(d1)

In [ ]:
# Open a data file with data separated by single-space, tab, or "any-space"
filename_s1 = "data/data2single.txt"
filename_t = "data/data2tab.txt"
filename_s = "data/data2.dat"
d2s1 = pd.read_csv(filename_s1, sep=' ', comment='#', names=['e','N'])
d2t = pd.read_csv(filename_t, sep='\t', comment='#', names=['e','N'])
d2 = pd.read_csv(filename_s, sep=r"\s+", comment='#', names=['e','N','hf'])
print(d2s1)
print("*"*25)
print(d2t)
print("*"*25)
print(d2)

In [ ]:
# Read only some of the data
d2 = pd.read_csv(filename_s, sep=r"\s+", comment='#', usecols=[0,1], names=['e','N'])
print(d2)

### 1.4 Read output files from MAESTRO
1. Ignore initial rows
2. Ignore rows at the bottom
3. Retrieve in-line informations
4. Correctly store data

In [ ]:
filename = "data/test.Spe"
# Ignore initial rows: skiprows=X with X=number of rows to skip (counting from the beginning of the file downwards)
# --> we need to find some reference that allows us to identify when the data starts
# --> in MAESTRO, the data are saved so that their value are found after a number of initial spaces
n_initial_rows = 0
with open(filename, 'r') as f:
    lines = f.read().splitlines()
    for line in lines:
        if "    " in line[:4]:
            break
        else:
           n_initial_rows += 1 # in Python x += 1 is a short command for x = x + 1

# Ignore bottom rows: nrows=X with X=number of rows to read
# --> We can either counts the number of lines with initial spaces
#     or find the number of footer-lines and subtract header-lines + footer-lines to the total number of lines -> read the file backwards using "for line in lines[::-1]:"
#     or retrieve info on the number of channels used: the datasets will have a number of rows/entries equal to the number of channels
# Retrieve in-line informations
# --> Find a reference that "introduces" the information sought: the line below "$DATA:" contains the first and last channel number
with open(filename, 'r') as f:
    lines = f.read().splitlines()
    for i, line in enumerate(lines):
        if "$DATA:" == line:
            ch_str = lines[i+1]
            ch_str_list = ch_str.split(" ")
            ch_initial = float(ch_str_list[0])
            ch_final = float(ch_str_list[1])
            break
n_rows_data = ch_final - ch_initial + 1
dm = pd.read_csv(filename, sep=" ", skiprows=n_initial_rows, nrows=n_rows_data, names=['N'])
print(dm)

In [ ]:
# Ignore initial spaces
dm = pd.read_csv(filename, sep=" ", skiprows=n_initial_rows, nrows=n_rows_data, skipinitialspace=True, names=['N'])
print(dm)
# Note: the best solution would be to use sep=r"\s+"
# --> this way you tell pandas "treat any space as a separator or something to skip if found before or after the numerical values


<div class="alert alert-success">
<h3><i class="fa fa-edit"></i> Exercise </h3>
    
Let's apply what we have just learnt on some real data!

$\gamma$-ray spectrometry with sodium iodide ($NaI$) detector:
1. Open the .Spe files you have collected during the previous laboratories: background, $^{60}Co$, $Na$, $^{241}Am$, $Ba$, $^{137}Cs$.
3. Create a DataFrame with two series of data: the channel number & the number of counts in each channel
4. Normalise the count-yield by the measurement time to make the different measurements comparable

# 2. Data visualisation - Matplotlib.pyplot
1. Plot histograms
2. Generic plot
3. Include the errorbar

### 2.1 Plot histograms

In [ ]:
# Let's use our datafile from MAESTRO
plt.figure() # This create a new 'empty' figure
# In order to plot our set of data as an histogram we need to define the bins' extremes and their centre
# Note: In MAESTRO, the channel values given are the lower extreme of each bin
ch_extr = np.arange(0, dm.shape[0]+1, step=1)
ch = (ch_extr[1:] + ch_extr[:-1]) / 2
dm['e'] = ch # Create a series in the dataframe for the energy in channels
ch_delta = ch_extr[1:] - ch_extr[:-1]
plt.bar(ch, dm['N'], width=ch_delta)
plt.xlabel("Channels [a.u.]")
plt.ylabel("Number of counts")

### 2.2 Generic plot

In [ ]:
plt.figure()
plt.plot(ch, dm['N'], label="line")
plt.plot(ch, dm['N'], '.', label="points")
plt.xlabel("Channels [a.u.]")
plt.ylabel("Number of counts")
plt.legend()

### 2.3 Include the errorbar

In [ ]:
# In a histogram-like plot
plt.figure()
plt.bar(d1['e'], d1['pd'], yerr=d1['pd_std'], width=np.diff(d1['e'])[0])
plt.xlabel("Channels [a.u.]")
plt.ylabel("Probability density")

# In a generic plot: plt.errorbar -> same properties as plt.plot but you can include xerr and yerr
plt.figure()
plt.errorbar(d1['e'], d1['pd'], yerr=d1['pd_std'], linestyle='-', marker='.')
plt.xlabel("Channels [a.u.]")
plt.ylabel("Probability density")

<div class="alert alert-success">
<h3><i class="fa fa-edit"></i> Exercise </h3>
    
Let's have a look on the $\gamma$-ray spectrometry data measured:
1. Plot the background.
2. Plot in different figures the spectra you have collected during the previous laboratories, including the background as a dashed line in each plot:$^{60}Co$, $Na$, $^{241}Am$, $Ba$, $^{137}Cs$.
3. Include the errorbars in at least one of the figure

# 3. Data analysis - Numpy & Scipy
1. Basic numpy functions and array manipulation
2. Fit a data-set with scipy.optimize.curve_fit

### 3.1 Basic numpy functions and array manipulation

In [ ]:
# Define an array in numpy
# Hard coded
a = np.array([1,2,3,4,5])
print("a: " + str(a))
# Array in numpy can be multi-dimensional: 2D array = matrix, but also 3D, 4D, ...
m = np.array([[1,2,3,4,5],[0,15,30,45,60]])
print("m: \n" + str(m))
# How can we check the size of an array? 
print("\na shape: " + str(a.shape))
print("m shape: " + str(m.shape))
# Initialise an array of zeros and assign the values later according to needs
b1 = np.zeros(5) # 1 dimension
print("b 1 dimension shape: " + str(b1.shape))
r = 3
c = 5
b2 = np.zeros((r,c)) # 2 dimensions
print("b 2 dimensions shape: " + str(b2.shape))
b4 = np.zeros((r,c,3,6)) # 4 dimensions
print("b 4 dimensions shape: " + str(b4.shape))
# we can also create an array of zero of the same 'shape' as an existing array
b = np.zeros(a.shape)
print("b shape: " + str(b.shape))
print("b: " + str(b))

# To assign the value to b we need to access b's elements --> in python the first element has index=0!
first = b[0]
third = b[2]
print(str("\nIn python the first element has index=0! b[0] = %.4f | b[2] = %.2e is its third element." % (first, third)))
# Assign a value
b[3] = 4
print(str("Now the fourth element of b is b[3] = %i " % (b[3])))

# Define an array of equally spaced values
vstart = 10
vend = 20
vstep = 2
clen = 5
c = np.arange(vstart, vend, step=vstep)
d = np.linspace(vstart, vend, num=clen)
dlog = np.logspace(vstart, vend, num=clen)
print("\nc: " + str(c) + " !!! when using np.arange(start,stop) the stop value is excluded! --> [start, stop)")
print("d: " + str(d))
print("dlog: " + str(dlog) + " !!! when using np.logspace, the start and stop given are the first and last exponent: [10^start , 10^stop]$")
dlog = np.logspace(-1, 5, num=7)
print("dlog re-defined: " + str(dlog))

# You can obtain a numpy array from pandas data-frames
theNcounts = dm['N'].to_numpy()
print("The theNcounts type is: " + str(type(theNcounts)))

# You can do any operation between array but bare in mind: in python +,-,*,/ are by default element-by-element so arrays must be of the same shape!
r = a + d
print("\nsum: " + str(r))
r = a - d
print("difference: " + str(r))
r = a * d
print("product: " + str(r))
r = a / d
print("ratio: " + str(r))
# square root, power, exponential and logarithmic functions
print("square root of c = " + str( np.sqrt(c) ))
print("c to the power of 4: method c**4 = " + str( c**4 ))
print("c to the power of 4: method np.pow(c,4) = " + str( np.pow(c,4) ))
print("exponential: e^c = " + str( np.exp(c) ))
print("logarithm natural = " + str( np.log(c) ))
print("logarithm (base 10) = " + str( np.log10(c) ))
# You can concatenate two arrays of any size, or add an element to an array
along = np.append(a, c)
along = np.append(along, 88)
print("\n along = " + str(along))
# You can create matrix of 1D arrays with the same size
acm_c = np.c_[a,c]
print("matrix created with array laid as columns:\n" + str(acm_c))
# You can access element from one array using logic-operation on the same array or on a different array with the same shape
print("The value of c corresponding to a value of a higher than 3 are: " + str( c[a>3] ))

# Numpy has many embedded functions to manipulate data
print("\nThe smallest number of counts larger than 0 is %i" % (np.min(theNcounts[theNcounts > 0])))
print("The largest number of counts is %.2e" % (np.max(theNcounts)))
rng = np.random.default_rng(2204)
norm_sample = rng.normal(30,5,1000)
print("The mean value and the standard deviation of the random generated sample are: %.2f and %.2f" % (np.mean(norm_sample), np.std(norm_sample)))

### 3.2 Fit a data-set using Scipy
Fitting data is a task that we cannot fulfil with Numpy. We need some specific tool. Scipy is a package that was created to perform most of scientific analysis tasks, such as fitting curves. A common fitting method can be found in the module "optimize": the curve_fit

1. Define a function in the correct format for the fitting
2. Fit the data-sample and obtain the fit-coefficients
3. Check the fit by comparing data and fitted curve

In [ ]:
#spo.curve_fit?

In [ ]:
# Define a function
def f(x):
    return x*2 + 1
f2 = lambda x: x*2 + 1
def test(x):
    if x > 0:
        print("Positive")
    elif x == 0:
        print("Null")
    else:
        print("Negative")
    return
thex = np.array([1, 0, -1])
print("f(x) : " + str(f(thex)))
print("f2(x) : " + str(f2(thex)))
for i, ix in enumerate(thex):
    print(str("\nx[%i] test: " % (i)))
    test(ix)

In [ ]:
# Gaussian fit
# Define function for fit: first input quantity is the variable given, the following will be fitted
def gauss(x, mu, sigma, weight):
    return weight * ( 1 / (sigma * np.sqrt(2*np.pi)) ) * np.exp(- 1/2 * ((x - mu) / sigma)**2)
# Let's fit a data-sample!
fit_coeff, cov_matrix = spo.curve_fit(gauss, dm['e'], dm['N'])
# The standard deviation of the parameters fitted can be found, as described in the curve_fit documentation
fc_err = np.sqrt(np.diag(cov_matrix))
print("The fit coefficients are returned as an array, with the coefficient in the order defined in the function.")
print("In our case, gauss(x,mu,sigma,weight), the order will be: mu (fit_coeff[0]), sigma (fit_coeff[1]), weight (fit_coeff[2]) = " + str(fit_coeff))
# Let's check the fit
plt.figure()
plt.plot(dm['e'], dm['N'], marker='.', linestyle='none', markersize=6, label='Experimental data')
ee = np.linspace(np.min(dm['e']), np.max(dm['e']), num=int(1e4))
Nfit = gauss(ee, *fit_coeff) # In this case, *fit_coeff passes the element of the array as single entries in the array order
plt.plot(ee, Nfit, marker='none', linestyle='-', label='Fitted curve')
plt.xlabel("Channels [a.u.]")
plt.ylabel("Number of counts")
plt.legend()

<h3><i class="fa fa-eye"></i> IMPORTANT! </h3>

<h4>Even though they are not strictly necessary, it is very important to give an initial guess and/or a range of values where to look for the parameters, in order to obtain a good fit! </h4>

In [ ]:
# Initial guess from rough eye-evaluation
mu0 = 20
sigma0 = 10
height0 = 80
w0 = height0 * sigma0 * np.sqrt(2*np.pi)
guess0 = np.array([mu0,sigma0,w0])
# The range of value where to look for the parameters might simply be: higher than 0
# Following curve_fit documentations they have to be given as a 2-ntuple : ( X1, X2 ) where X1 is an array with lower bounds and X2 is an array with upper bounds
param_range = (np.zeros(3), np.zeros(3) + np.inf)

# Fit passing initial guess and bounds
fit_coeff, cov_matrix = spo.curve_fit(gauss, dm['e'], dm['N'], p0=guess0, bounds=param_range)
fc_err = np.sqrt(np.diag(cov_matrix))
print("The fit coefficients are returned as an array, with the coefficient in the order defined in the function.")
print("In our case, gauss(x,mu,sigma,weight), the order will be: mu (fit_coeff[0]), sigma (fit_coeff[1]), weight (fit_coeff[2]) = " + str(fit_coeff))

# Let's check the fit again
plt.figure()
plt.plot(dm['e'], dm['N'], marker='.', linestyle='none', markersize=6, label='Experimental data')
ee = np.linspace(np.min(dm['e']), np.max(dm['e']), num=int(1e4))
Nfit = gauss(ee, *fit_coeff) # In this case, *fit_coeff passes the element of the array as single entries in the array order
plt.plot(ee, Nfit, marker='none', linestyle='-', label="Fitted curve \n" + str(r"$\mu$=%.2f & $\sigma$=%.2f" % (fit_coeff[0], fit_coeff[1])))
plt.xlabel("Channels [a.u.]")
plt.ylabel("Number of counts")
plt.legend()

<div class="alert alert-success">
<h3><i class="fa fa-edit"></i> Exercise </h3>
    
Let's do some basic analysis on the $\gamma$-ray spectrometry data measured. For the different data measured ($^{60}Co$, $Na$, $^{241}Am$, $Ba$, $^{137}Cs$):
1. Subtract the background and plot the cleaned spectra.
2. Visually identify the peaks and fit the appropriate subset of data with a Gaussian function.
3. Plot the cleaned spectrum and the fitted peaks to check the fitting procedure worked well.
4. Retrieve and print the peaks position in channels
5. Assigning to each peak its nominal energy value, plot the calibration curve

<h4><i class="fa fa-lightbulb"></i> Suggestion </h4>
Simultaneously analysing all measurements can be overwhelming. Do the exercise for one of the measurement (e.g. the $^{60}Co$ dataset) and execute it. Once you becomes practical with the analysis, you can re-iterate it on the other dataset by simply changing the data-frame analysed and the fit intervals.